# Cellpose in Python

<div class="custom-button-row">
    <a 
        class="custom-button custom-download-button" href="../../../notebooks/05_segmentation/deep_learning/cellpose_notebook.ipynb" download>
        <i class="fas fa-download"></i> Download this Notebook
    </a>
    <a
    class="custom-button custom-download-button" href="https://colab.research.google.com/github/bobiac/bobiac-book/blob/gh-pages/colab_notebooks/05_segmentation/deep_learning/cellpose_notebook_colab.ipynb" target="_blank">
        <img class="button-icon" src="../../../_static/logo/icon-google-colab.svg" alt="Open in Colab">
        Open in Colab
    </a>
</div>

In [ ]:
# /// script
# requires-python = ">=3.12"
# dependencies = [
#     "matplotlib",
#     "cellpose",
#     "tqdm"
# ]
# ///

## Overview

[Website](https://www.cellpose.org) | [GitHub](https://github.com/mouseland/cellpose) | [Paper](https://www.biorxiv.org/content/10.1101/2025.04.28.651001v1) | [Cellpose API](https://cellpose.readthedocs.io/en/latest/api.html#)

In this section, we’ll learn how to use **Cellpose**, a powerful deep learning tool for cell segmentation, works on a wide variety of microscopy images and doesn’t require retraining for many common use cases.

In this notebook, we’ll see how to run Cellpose on single images or on a folder of images, and how to visualize and save the results.

<p class="alert alert alert-info">
    <strong>💡 Tip:</strong> If you're not using a GPU (or are on a Mac with Apple Silicon), we recommend running this notebook on <a href="https://colab.research.google.com/github/bobiac/bobiac-book/blob/gh-pages/colab_notebooks/05_segmentation/deep_learning/cellpose_notebook_colab.ipynb" target="_blank"> Google Colab</a> for faster performance.
</p>

## Import Libraries

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from cellpose import core, io, models, plot
from tqdm import tqdm

## Setup

In [ ]:
io.logger_setup()  # to get printing of progress

use_gpu = core.use_gpu()
print("GPU available:", use_gpu)

## Run Cellpose on a Single Image

In this section, we’ll apply Cellpose to a single image and visualize the segmentation result.

### Load the Image

To load the image, we can use the `imread` method from the Cellpose `io` module (or the `tifffile` library if you add it to the dependencies):

In [ ]:
image_path = "../../../_static/images/cellpose/cell_cellpose.tif"
image = io.imread(image_path)  # or image = tifffile.imread(image_path)

print(image.shape)

### Initialize the Model

To initialize Cellpose model we can use the `models.CellposeModel()` class.

There are other parameters we can set when initializing the model, here we will only use `pretrained_model` to specify which pretrained model to use (e.g. the default `cpsam` model or a custom trained model) and `gpu` to specify whether to use GPU (if available) for faster inference.

<p class="alert alert alert-info">
    <strong>Note:</strong> If it is the first time you run this notebook, the model will be downloaded automatically. This may take a while.
</p>

In [ ]:
model = models.CellposeModel(pretrained_model="cyto3", gpu=use_gpu)

### Run Cellpose

After initializing the model, we can run it on the image using the `model.eval()` method (see dropdown below for parameters details).

:::{dropdown} CellposeSam `model.eval()` Parameters

```python
model.eval(
    x,
    channel_axis=None,
    normalize=True,
    invert=False,
    diameter=None,
    flow_threshold=0.4,
    cellprob_threshold=0.0,
    min_size=15,
    max_size_fraction=0.4,
    niter=None,
    compute_masks=True,
    batch_size=8,
    resample=True,
    bsize=256,
    tile_overlap=0.1,
    augment=False,
    do_3D=False,
    z_axis=None,
    anisotropy=None,
    flow3D_smooth=0,
    stitch_threshold=0.0,
)
```

**Input**

| Parameter | Default | Description |
|---|---|---|
| `x` | — | A single image or list of images (2D/3D/4D). For `cpsam`, up to 3 channels are used. |
| `channel_axis` | `None` | Which axis of `x` is the channel axis. If `None`, Cellpose auto-detects it. |

**Preprocessing**

| Parameter | Default | Description |
|---|---|---|
| `normalize` | `True` | Normalize intensities to 0–1 using 1st/99th percentile. Pass `False` to skip, or a dict for fine-grained control (see below). |
| `invert` | `False` | Invert pixel intensities before running the network. Useful for brightfield images where cells are dark on a bright background. |
| `diameter` | `None` | Expected cell diameter in pixels. Used to rescale the image so cells appear ~30 px wide to the model. If `None`, no rescaling is applied. |

When passing `normalize` as a dict, all keys are optional and can be combined:

```python
# Tile-based normalization: useful when illumination is uneven across the image.
# The image is split into blocks of ~100 px and each block is normalized independently.
normalize = {"tile_norm_blocksize": 100}

# Fixed intensity range: skip percentile computation and clamp to known values.
normalize = {"lowhigh": [200, 4000]}

# Custom percentiles instead of the default 1st/99th.
normalize = {"percentile": [5, 95]}

# Sharpen before segmenting (value ≈ 1/4 to 1/8 of cell diameter in px).
normalize = {"sharpen": 5}

# Keys can be combined — e.g. tile normalization + custom percentiles:
normalize = {"tile_norm_blocksize": 100, "percentile": [2, 98]}
```

**Segmentation quality**

| Parameter | Default | Description |
|---|---|---|
| `flow_threshold` | `0.4` | Maximum allowed flow error for a mask to be kept. **Increase** to recover more masks (or set to `0.0` to keep all); **decrease** to discard ill-shaped masks. Not used in 3D. |
| `cellprob_threshold` | `0.0` | Minimum cell probability for a pixel to be included in a mask. **Decrease** to find more/larger masks; **increase** to suppress dim or spurious detections. |
| `min_size` | `15` | Minimum mask area in pixels. Smaller objects are discarded. |
| `max_size_fraction` | `0.4` | Masks larger than this fraction of the total image area are removed. |
| `niter` | `None` | Number of iterations for the flow integration step. If `None`, set automatically proportional to `diameter`. |
| `compute_masks` | `True` | If `False`, skips flow integration and returns empty masks (useful to get flows/styles only). |

**Performance**

| Parameter | Default | Description |
|---|---|---|
| `batch_size` | `8` | Number of 256×256 tiles processed per GPU forward pass. Increase for faster throughput (uses more GPU memory); decrease if you hit out-of-memory errors. |
| `resample` | `True` | Run flow dynamics at the original image resolution. Slower but produces more accurate boundaries. |
| `bsize` | `256` | Tile size used by the network. Keep at 256 (matches training). |
| `tile_overlap` | `0.1` | Fractional overlap between adjacent tiles. Helps avoid boundary artifacts. |
| `augment` | `False` | Tile the image with overlapping tiles and flip augmentation. Slightly more accurate but slower. |

**3D segmentation**

| Parameter | Default | Description |
|---|---|---|
| `do_3D` | `False` | Set to `True` to run full 3D segmentation on a Z-stack. |
| `z_axis` | `None` | Which axis is the Z axis (for 3D images). If `None`, auto-detected. |
| `anisotropy` | `None` | Z-to-XY voxel size ratio (e.g. `2.0` if Z is sampled at half the XY density). Used to rescale the Z axis before 3D segmentation. |
| `flow3D_smooth` | `0` | Smooth 3D flows with a Gaussian filter of this stddev. Helps reduce Z-fragmentation and ring artifacts. Can be a list `[z, y, x]` for axis-independent smoothing. |
| `stitch_threshold` | `0.0` | If `> 0` and `do_3D=False`, stitch 2D masks across Z slices into a 3D volume. |

**Returns**

| Output | Description |
|---|---|
| `masks` | 2D label array (or list of them). `0` = background; `1, 2, …` = individual cell IDs. |
| `flows` | List of flow outputs per image: `flows[0]` = RGB flow visualization; `flows[1]` = XY flow vectors; `flows[2]` = cell probability map. |
| `styles` | Style vectors (legacy, all zeros for `cpsam`). |

:::

In [ ]:
masks, flows, styles = model.eval(image)

### Display the Results

To display the results, we can use the `show_segmentation` method from the Cellpose `plot` module that will show the original image, predicted masks, outlines, and flow fields in a single figure (alternatively, you can use other libraries like `ndv` or `matplotlib` to directly visualize the outputs).

In [ ]:
fig = plt.figure(figsize=(12, 5))
plot.show_segmentation(fig, image, masks, flows[0])
plt.tight_layout()
# Optional if you want to also save the figure
# plt.savefig(f"path/to/output/{Path(image_path).stem}_cp_output.png")
plt.show()

<div align="left"> <img src="https://raw.githubusercontent.com/bobiac/bobiac-book/main/_static/images/cellpose/cellpose_out.png" alt="Ilastik Logo" width="1000"></div>

To save the labelled masks as a .tif file, you can use the Cellpose `imsave` method from the `io` module (or e.g. the `tifffile` library if you add it to the dependencies):

In [ ]:
output_path = f"path/to/output/{Path(image_path).stem}_labels.tif"
io.imsave(output_path, masks)  # or tifffile.imwrite(output_path, masks)

To override the defaults, pass any parameter explicitly to `model.eval()` (see hidden cell above). For example, to adjust the segmentation quality parameters:

```python
masks, flows, styles = model.eval(
    image,
    flow_threshold=0.2,
    cellprob_threshold=0.8,
    min_size=800,
)
```

Another example is if you want to run 3D segmentation on a z-stack. In this case you can set `do_3D=True` and specify the `z_axis` and `anisotropy` if needed:

```python
masks, flows, styles = model.eval(
    image,
    do_3D=True,
    z_axis=0,
    anisotropy=2.0,
)
```

## Run Cellpose on a Folder of Images

Now that we’ve seen how to run Cellpose on a single image, let’s scale up and apply it to a **folder of images**. This is useful when you have an entire experiment or dataset that you want to segment automatically.

There are few different ways to do this (see Bonus sections [1](#bonus-1-batch-processing-and-ram-management) and [2](#bonus-2-timelapse-batch-processing)), but the simplest one is to just **loop through the images in the folder** and run `model.eval()` on each one. During each iteration, we can save the predicted masks to an output folder using the `imsave` method from the Cellpose `io` module (or the `tifffile` library if you add it to the dependencies).

In [ ]:
# path to the folder containing the images to segment
folder_path = Path("data/05_segmentation_cellpose")

# Get the sorted list of all .tif images in the folder
images_path = sorted(folder_path.glob("*.tif"))

# Run Cellpose on each image one by one
# NOTE: tqdm is used to show a progress bar, but you can remove it if you don't want it
for image_path in tqdm(images_path, desc="Processing images"):
    # Load the image
    image = io.imread(image_path)
    # Run Cellpose on the image
    masks, flows, styles = model.eval(image)
    # Save the segmentation results as a TIFF file
    output_path = folder_path / f"{image_path.stem}_labels.tif"
    io.imsave(output_path, masks)  # or tifffile.imwrite(output_path, masks)

## Bonus 1: Batch Processing and RAM Management

The code above is best if you have to segment big size images that you must open one at a time to fit in memory.

If you have many smaller images that if loaded together they fit in memory, it is more efficient to load them all at once and pass the list of images to `model.eval()`. In this case, Cellpose will process them in batches.

Let’s say for example that you can open 3 images at a time in memory from a folder. You can load them in batches of 3 and pass each batch to `model.eval()`.

In [ ]:
# path to the folder containing the images to segment
folder_path = Path("data/05_segmentation_cellpose")

# Get the sorted list of all .tif images in the folder
images_path = sorted(folder_path.glob("*.tif"))

# Group the image paths into batches of n images (e.g., 3)
batch_size = 3  # number of images per batch
batches = [
    images_path[i : i + batch_size] for i in range(0, len(images_path), batch_size)
]
# which is equivalent to the following code:
# batches = []
# for i in range(0, len(images_path), batch_size):
#     batches.append(images_path[i:i + batch_size])

# Run Cellpose on images in batches of 3
# NOTE: tqdm is used to show a progress bar, but you can remove it if you don't want it
for batch in tqdm(batches, desc="Processing batches"):
    # Load all the images in the batch
    images = [io.imread(image_path) for image_path in batch]
    # Run Cellpose on the batch of images
    batch_masks, batch_flows, batch_styles = model.eval(images)
    # Save the segmentation results for each image in the batch
    for image_path, mask in zip(batch, batch_masks):
        output_path = folder_path / f"{image_path.stem}_labels.tif"
        io.imsave(output_path, mask)  # or tifffile.imwrite(output_path, mask)
    # which is equivalent to:
    # for i in range(len(batch)):
    #     output_path = folder_path / f"{batch[i].stem}_labels.tif"
    #     io.imsave(output_path, batch_masks[i])

If you are aware of your GPU specs and memory limits, you can also adjust the `batch_size` parameter in `model.eval()` to optimize performance and using the memory you have available. Just be careful not to set it too high or you might get out-of-memory errors.

As explained above in the `model.eval()` parameters ([dropdown](#run-cellpose) above), the `batch_size` parameter controls how many 256x256 tiles are processed per GPU forward pass. This means that if you have a 512x512 pixel image, it will be split into 4 tiles of 256x256.

If you set `batch_size=4`, for example, it means that the ...

## Bonus 2: Timelapse Batch Processing

When working with a timelapse, you cannot pass the raw stack array directly to `model.eval()`, Cellpose has no concept of timepoints and would misinterpret the time axis as channels or a Z-stack.

The solution is to convert the stack into a **list of frames** using `list(stack)`, which slices along axis 0. The result depends on your axis order:

```python
# TCYX — axis 0 is time → list gives T frames of shape (C, Y, X)
stack.shape  # (10, 2, 512, 512)
list(stack)  # → [frame_0, ..., frame_9], each (2, 512, 512)
```

So we can pass the list of frames directly to `model.eval()`:

```python
frame_masks, frame_flows, frame_styles = model.eval(list(stack))
```

`frame_masks` will be a list of 2D label arrays, one per timepoint.


If the time axis is not the first one (e.g. CTYX), you need to transpose the stack first to get it into TCYX order before converting to a list:

```python
# CTYX — axis 0 is channel → wrong, need to transpose first
stack.shape  # (2, 10, 512, 512)
stack = stack.transpose(1, 0, 2, 3)  # CTYX → TCYX, then list(stack) works
```

Now that the stack is in TCYX order, you can pass it as a list to `model.eval()` and Cellpose will process each frame independently, returning one mask per frame.

```python
frame_masks, frame_flows, frame_styles = model.eval(list(stack))
```